# IGNITE shot designer

Type what you want to do with a DIII-D discharge and send it to Claude. **Only your prompt
leaves the cluster**, together with fixed instructions that describe IGNITE's actuator
inputs and the plan language (`designer/PROMPT_ONLY.md`, no DIII-D data in it). Claude has
no tools and no data access: it designs the actuator trajectories from your prompt and its
own knowledge of tokamak physics and DIII-D, and returns a plan, what it expects, and the
checks it proposes.

Everything measured stays here and is used only for the **local cross-check**: the plan is
built on the reference shot's measured actuators and tested against what DIII-D has
delivered, precedents are searched with Claude's conditions and the logbook with its query.
These results are shown to you and never sent to Claude. To revise, edit your prompt in
your own words (e.g. "210R above its limit, use the fill split") and send again; tick the
box to include Claude's previous plan (its own output).

```
your prompt ──► Claude (no tools, no data) ──► plan + expectation + proposed checks
                                                     │
          local, on the cluster only:  build on measured actuators ─► database check
                                       precedents, logbook ─► shown to you
                                                     │
                       Export plan ─► Simulate plan (GPU) ─► IGNITE rollouts
```

- **Kernel:** *FusionAIHub (stellarai)*. If missing:
  `pixi run --frozen -e stellarai python -m ipykernel install --user --name fusionaihub-stellarai`.
- **Claude:** through Claude Code (`claude` installed and logged in on the machine running
  this kernel), run from an empty directory with every tool, MCP server and setting off
  (`gui.claude_command`). The card *Sent to Claude* shows exactly what went out. About
  30 s and $0.10 per request.
- **Nothing runs on the GPU** until you press *Simulate plan* (about 10-20 min per seed).
- Every request's transcript is saved in `designer_runs/_agent_logs/`; the exported plan
  carries Claude's answer (`claude_answer.json`).

## 1. Design with Claude

In [ ]:
%matplotlib inline
from tokamak_foundation_model.ignite.designer import gui, tools

gui.agent()   # edit the prompt, press "Send to Claude"

## 2. What the local tools compute, step by step

The paper's design (`designer_runs/agent_204011_rotation_v5_maskfree`), taken apart: each
local tool's input and output and the arithmetic in between, down to the actuator arrays
IGNITE receives. All of it runs on the cluster; none of it is sent to Claude.

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from tokamak_foundation_model.ignite.designer import tools, plan as pl, machine as mc, shots, simulate as sim

RUN = Path("/scratch/gpfs/EKOLEMEN/ps9551/ignite/models/designer_runs/agent_204011_rotation_v5_maskfree")
SHOT, T_CHANGE = 204011, 3.5
show = lambda x: print(json.dumps(x, indent=1, default=str))

### Step 1: what the database holds for this era (`machine`)
The actuator channels, the frame grid (50 ms frames from 1.0 s)
and, per beam source, what the 2000-shot block around 204011 shows: whether it ran, its
maximum power, its torque per MW in each direction, and the most it delivered in any shot.

In [ ]:
m = tools.machine(SHOT)
print(list(m))
show({b: v for b, v in m["era"]["beams"].items()} if "era" in m else m)

### Step 2: the measured shot (`describe_shot`)
Every beam source over the window to be changed (3.5 s to the end of the beams): power,
voltage, torque and torque per MW with its direction; gyrotrons, gas, coils; and the core
plasma quantities. Here: six co-current sources.

In [ ]:
show(tools.describe_shot(SHOT, t0=T_CHANGE, t1=5.8))

### Step 3: the logbook (`logbook`, `search_logbook`)
What 204011 was meant to do and what happened, then other shots that mention low torque or
counter-injection (try several wordings, since operators write them differently).

In [ ]:
lb = tools.logbook(SHOT, max_chars=3000)
show(lb)

In [ ]:
for q in ('"low torque" OR "zero torque" OR "balanced torque"', '"counter NBI" OR "counter-NBI" OR "counter beam"'):
    r = tools.search_logbook(q, limit=8)
    print(q); show(r)

### Step 4: measured precedents (`search_precedents`)
Shots in which the net beam torque stayed below 1 N m for at least 0.5 s while beams were on,
ranked by similarity to 204011.

In [ ]:
show(tools.search_precedents(conditions=["torque_Nm < 1", "pnbi_MW > 4"], window=(2.0, 6.0),
                             min_duration_s=0.5, near=SHOT, limit=10))

### Step 5: torque per MW of every source (`plan.torque_per_mw`)
Measured in this shot for a source that ran: the median of torque / power over the frames
it was on. For a source that was off (210L, 210R): the median over the era's shots in which
it injected counter-current. These are the numbers the balance uses.

In [ ]:
s = shots.load(SHOT)
for b in mc.BEAMS:
    try:
        print(f"{b}: {pl.torque_per_mw(s, b):+.3f} N m/MW")
    except ValueError:
        print(f"{b}: counter {pl.torque_per_mw(s, b, 'counter'):+.3f} N m/MW (off in this shot)")

### Step 6: the options (`beam_options`)
Every set of other sources to switch off that reaches the target with 210L/210R
counter-current and 30L/330L (the main CER views) kept, with the counter power it needs and
whether every source stays within what it delivered in the database. First a
0.5 N m target ("fill": the source with more torque per MW, 210R, runs up to its limit and
210L takes the rest; "equal_power": both at the same power), then the chosen 0.9 N m.

In [ ]:
for target, split in ((0.5, "fill"), (0.5, "equal_power"), (0.9, "equal_power")):
    o = tools.beam_options(SHOT, T_CHANGE, counter=["NB21L", "NB21R"], keep_on=["NB30L", "NB33L"],
                           target_Nm=target, t_end=5.8, split=split, limit=4)
    print(f"target {target} N m, split {split}: {o['n_feasible']} feasible of {o['n_options']}")
    for opt in o["options"]:
        print({k: v for k, v in opt.items() if k not in ("ops", "at_max_fraction")})

### Step 7: the plan
The chosen plan, saved as `plan.json`: one `balance_torque` operation.

In [ ]:
plan = json.loads((RUN / "plan.json").read_text())
show(plan)

### Step 8: what `balance_torque` computes, frame by frame
From 3.5 s: (1) 150L, 150R, 30R and 330R are set to zero power, voltage and torque;
(2) the kept sources (30L, 330L) keep their measured waveforms and torque;
(3) the counter torque needed in each frame is `need = torque(kept) - 0.9 N m`;
(4) with equal power, both 210s run at `p = need / (|r_210L| + |r_210R|)`, and each injects
`r * p`; (5) their voltage is set to their typical voltage. Below the same arithmetic done by
hand and checked against the tool.

In [ ]:
P, T, V = mc.columns("pinj"), mc.columns("tinj"), mc.columns("beam_voltage")
k = np.arange(s.act.shape[0]) >= mc.frame_of(T_CHANGE)
beams = mc.time_of(np.flatnonzero(k)) < 5.85   # until the measured beams end
op = plan["ops"][0]
idx = {b: mc.BEAMS.index(b) for b in mc.BEAMS}
kept = [idx[b] for b in mc.BEAMS if b not in op["off"] and b not in op["counter"]]
r = [pl.torque_per_mw(s, b, "counter") for b in op["counter"]]
need = np.clip(s.act[k, T][:, kept].sum(1) - op["target_Nm"], 0, None)
p_mw = need / sum(abs(x) for x in r)

built = pl.build(plan)
for b, rate in zip(op["counter"], r):
    tool = built.act[k, P.start + idx[b]] * 1e-6
    print(f"{b}: rate {rate:+.3f} N m/MW, power by hand vs tool: max |diff| {np.abs(tool - p_mw).max():.2e} MW, "
          f"mean {p_mw[beams].mean():.2f} MW, max {p_mw[beams].max():.2f} MW over 3.5-5.85 s")
t = mc.time_of(np.flatnonzero(k))
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(t, s.act[k, T][:, kept].sum(1), label="torque of kept 30L+330L")
ax[0].plot(t, need, label="counter torque needed"); ax[0].set_ylabel("N m"); ax[0].legend()
ax[1].plot(t, p_mw, label="each 210 source"); ax[1].set_ylabel("MW"); ax[1].legend()
for a in ax: a.set_xlabel("time (s)")

### Step 9: every actuator channel, measured and planned
All channels the plan changes (power, voltage and torque per source), with their means over
the changed frames, then the totals before and after.

In [ ]:
show(built.summary())
from tokamak_foundation_model.ignite.designer import gui
gui.plot_actuators(built);

### Step 10: the checks (`check_plan`)
Physical consistency and limits: every source against the most it delivered in this era and
in the whole database; here 210L runs above its era maximum but within what it delivered in earlier
shots.

In [ ]:
show(tools.check_plan(plan))

### Step 11: what IGNITE receives
The actuators per 50 ms frame (88 channels), normalized with the training set's statistics,
`(raw - mean) / std`, exactly like every training shot. `export_plan` writes two inputs:
`plan.npz` holds the plan (`act`, and `act_norm` normalized), and `act_global/204011.pt` holds
the shot's measured actuators. The replay rolls out with the measured ones; the plan rollout
replaces them by `plan.npz` (`--actuator_mode file:<run>/plan.npz`), so both differ only
where the plan changes something.

In [ ]:
import torch
z = np.load(RUN / "plan.npz")
print({k_: z[k_].shape for k_ in z.files})
print("plan.npz act == built plan:", np.allclose(z["act"], built.act))
print("plan.npz act_norm == built plan, normalized:", np.allclose(z["act_norm"], built.normalised(), atol=1e-3))
g = torch.load(RUN / "act_global" / f"{SHOT}.pt", weights_only=False)["actuators_global"].float().numpy()
n = built.normalised()
print("measured (act_global) == plan on unchanged frames:", np.allclose(g[~built.changed], n[~built.changed], atol=1e-3))
print("frames x channels the plan changes:", int(built.changed.sum()), "from", mc.time_of(np.flatnonzero(built.changed.any(1))[0]), "s")

### Step 12: the simulation command
What `simulate(..., confirm=True)` submits: `run_shot.sh` rolls out the measured actuators
(replay) and the plan (`--actuator_mode file:<run>/plan.npz`) from the same initial state
(1-2 s), for each seed. The paper's rollouts used the maskfree soup model, 8 seeds.

In [ ]:
print(" ".join(sim.command(RUN, seeds=range(1, 9))))
print(sorted(p.name for p in (RUN / "soup").iterdir()))

### Step 13: the result
Core |v_tor| (CER T02) and T_i (CER T01), measured, replay (measured actuators) and plan,
mean of the 8 rollouts, and the plan-minus-replay change over 4.5-5.0 s.

In [ ]:
d = np.load("/scratch/gpfs/EKOLEMEN/ps9551/ignite/paper_figures/data8_maskfree/designer_204011_data.npz")
tt = mc.T0 + mc.FRAME_S * np.arange(d["pred__cer_rot"].shape[1]); w = (tt >= 4.5) & (tt < 5.0)
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
for a, (n, c, sc, lab) in zip(ax, (("cer_rot", 1, 1.0, "|v_tor| (km/s)"), ("cer_ti", 0, 1e-3, "T_i (keV)"))):
    f = (lambda x: np.abs(x)) if n == "cer_rot" else (lambda x: x)
    for which, col in (("gt", "C0"), ("real", "C1"), ("pred", "C4")):
        x = f(d[f"{which}__{n}"][:, :, c] * sc).mean(0)
        a.plot(tt, x, color=col, label={"gt": "measured", "real": "replay", "pred": "plan"}[which])
        if which != "gt":
            print(f"{n} {which}: {np.nanmean(x[w]):.2f} over 4.5-5.0 s")
    a.axvline(T_CHANGE, ls=":", color="0.5"); a.set_ylabel(lab); a.set_xlabel("time (s)"); a.legend()

## 3. Design by hand

The same tools without the agent: look at a shot, find precedents, enumerate beam sets, write
and check a plan, export and simulate it.

In [ ]:
%matplotlib inline
from tokamak_foundation_model.ignite.designer import gui, tools

gui.app(204011)

## The same steps as code

The choices below (which sources inject counter-current, which must stay on) are the
designer's, i.e. yours or the agent's; the tools only measure and check them against the
database. Here: zero net beam torque from 3.5 s in 204011 with NB21L/NB21R counter-current
and NB30L/NB33L kept on. `beam_options` lists every set of other sources to switch off; for a
set that needs more counter power than the database shows was ever run, it gives the largest
fraction of the kept sources' power at which it stays within the database.

In [ ]:
opts = tools.beam_options(204011, 3.5, counter=["NB21L", "NB21R"],
                          keep_on=["NB30L", "NB33L"], target_Nm=0.0)
print(opts["n_feasible"], "feasible of", opts["n_options"])
best = opts["options"][0]
best

In [ ]:
plan = {"shot": 204011,
        "goal": "zero net beam torque from 3.5 s",
        "rationale": "NB21L/R counter-current, NB30L/NB33L kept (power reduced where needed)",
        "ops": best["ops"] if best["feasible"] else best["at_max_fraction"]["ops"]}
check = tools.check_plan(plan)
print(check["ok"]); check["findings"]

In [ ]:
from tokamak_foundation_model.ignite.designer import plan as pl
gui.plot_actuators(pl.build(plan));

In [ ]:
# Writes the IGNITE input; prints the sbatch command. Nothing is submitted.
exported = tools.export_plan(plan, name="example_204011_zerotorque")
exported

In [ ]:
# GPU rollout (about 10-20 min per seed). Only with your approval:
# tools.simulate(exported["run_dir"], seeds=[1, 2, 3, 4], confirm=True)
# tools.simulation_status(exported["run_dir"])
# gui.plot_results(exported["run_dir"]); tools.simulation_results(exported["run_dir"])